##### Extracting and ingesting the data from data.sec.gov

The U.S. Securities and Exchange Comission allows HTTP requests for obtaining mandatory disclosures filed by every U.S. public company. Things like quarterly revenue, institutional holdings or 8-K events can be obtained as a JSON (or XBRL for financial facts) through the SEC's structured data API at data.sec.gov. The request just needs to comply with some rules:
- Instead of tickers, companies are identified by their Central Index Key (CIK), which can be obtained through the company_tickers.json available at www.sec.gov/files/company_tickers.json. Moreover, these have to be 10 digits long and padded with zeroes. For example, AAPL becomes CIK0000320193.
- There's a mandatory header for submitting these requests which specifies a name and an email. If omitted, the request will return a 403: Forbidden error.
- A rate limit of 10 requests per second. For good measure, we'll send about 8 by simply adding time.sleep(0.12) between calls.

For these reasons, we'll prepare the header beforehand. We'll also prepare some parameters for the rate limit and a historical data limit:

In [1]:
import requests
import time
from databricks.connect import DatabricksSession
import os
import json
import io
import re

catalog_name = "financial_db"
schema_name = "filings"

HEADERS = {"User-Agent": "SentimentAnalysisRAG miprie04@ucm.es"}
TICKERS = {"AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA"}
TABLE = f"{catalog_name}.{schema_name}.sec_rag_bronze"
RATE_LIMIT_DELAY = 0.12 # ~8 requests per second
YEARS_LIMIT = 5 # Limit to the last 5 years of filings

And we'll set up the Unity Catalog Infrastructure for Databricks. We'll download the raw files into this volume and later on copy them into our bronze table. This ensures the raw files are reusable if something happens and improves efficiency, instead of rewriting the whole table every time a new file is added:

In [10]:
os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
os.environ.pop("DATABRICKS_AUTH_TYPE", None) # This fails otherwise. Maybe I'll get a PAT later

spark = DatabricksSession.builder.getOrCreate()

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE} (
    ticker STRING,
    cik STRING,
    date STRING,
    accession_number STRING,
    url STRING,
    raw STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.appendOnly' = 'true',
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
);
''')

""


Now, we'll fetch the CIKs we are interested in, which we will obtain through the company_tickers JSON. With this, we can get the 10-K files associated with each company. Form 10-K is a comprehensive annual report required by the SEC that details a publicly traded company's financial performance, business operations, risks, and audited financial statements for the prior fiscal year. More information about the 10-K can be found at www.sec.gov/files/reada10k.pdf.

We'll fetch all the 10-Ks in the last YEARS_LIMIT years:

In [11]:
def get_ciks():
    url = "https://www.sec.gov/files/company_tickers.json"

    time.sleep(RATE_LIMIT_DELAY) # SEC limit
    res = requests.get(url, headers=HEADERS, timeout=10)
    if res.status_code != 200:
        return None
    data = res.json()

    return {v["ticker"]: str(v["cik_str"]).zfill(10) for v in data.values()} # Map of tickers with their formatted CIK


def fetch_10ks(cik: str, limit: int = YEARS_LIMIT):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    time.sleep(RATE_LIMIT_DELAY) # SEC limit
    res = requests.get(url, headers=HEADERS, timeout=10)
    if res.status_code != 200:
        return None

    results = [] # Array for storing the filings data, we'll do the requests later
    counter = 0
    
    filings = res.json()["filings"]["recent"]
    for i, form in enumerate(filings["form"]):
        if form == "10-K":
            accession = filings["accessionNumber"][i]
            doc_name = filings["primaryDocument"][i]
            url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}/{doc_name}"
            results.append({
                "accession_number": accession,
                "date": filings["filingDate"][i],
                "url": url
            })
            counter += 1
            if counter >= limit:
                return results
    return results

def get_bronze():
    ciks = get_ciks()
    existing_df = spark.sql(f"SELECT accession_number FROM {TABLE}")
    existing = set(v["accession_number"] for v in existing_df.collect())
    batch = []

    for ticker in TICKERS:
        cik = ciks.get(ticker)
        if not cik:
            continue

        doc_info = fetch_10ks(cik, 1)
        for filing in doc_info:
            if filing["accession_number"] not in existing:
                time.sleep(RATE_LIMIT_DELAY) # SEC limit
                res = requests.get(filing["url"], headers=HEADERS, timeout=30)
                if res.status_code==200:
                    res_cleaned = re.sub(r'data:image/[^;]+;base64,[A-Za-z0-9+/=]+', '', res.text) # Remove images from the document to reduce size
                    batch.append({
                        "ticker": ticker,
                        "cik": cik,
                        "date": filing["date"],
                        "accession_number": filing["accession_number"],
                        "url": filing["url"],
                        "raw": res_cleaned
                    }) # The data we'll have in our table
                else:
                    print(f"Error fetching document")
    if batch:
        df = spark.createDataFrame(batch)
        df.write.format("delta").mode("append").saveAsTable(TABLE)

get_bronze()

In [12]:
spark.sql(f'''
SELECT ticker, cik, date, accession_number, url, length(raw) AS charCount FROM {TABLE}
ORDER BY ticker ASC, date DESC
''').toPandas()

,ticker,cik,date,accession_number,url,charCount
0,AAPL,0000320193,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/000032...,1520208
1,AMZN,0001018724,2026-02-06,0001018724-26-000004,https://www.sec.gov/Archives/edgar/data/000101...,1968342
2,GOOGL,0001652044,2026-02-05,0001652044-26-000018,https://www.sec.gov/Archives/edgar/data/000165...,2616499
3,MSFT,0000789019,2026-07-29,0001193125-26-323660,https://www.sec.gov/Archives/edgar/data/000078...,8585501
4,NVDA,0001045810,2026-02-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/000104...,1967816
5,TSLA,0001318605,2026-01-29,0001628280-26-003952,https://www.sec.gov/Archives/edgar/data/000131...,2391529
